# 차트, 그래프 및 슬라이드 자료 다루기
Claude는 차트, 그래프 및 더 폭넓은 슬라이드 자료를 다루는 능력이 매우 뛰어납니다. 사용 사례에 따라 활용할 수 있는 여러 팁과 기법이 있습니다. 이 recipe에서는 Claude를 이러한 자료와 함께 사용하는 일반적인 패턴을 보여줍니다.

## 차트와 그래프
대부분의 경우 claude를 차트 및 그래프와 함께 사용하는 방법은 간단합니다. 이를 가져와 Claude에 전달하는 방법과 결과를 개선하는 몇 가지 일반적인 팁을 살펴보겠습니다.

### 자료 수집 및 Claude API 호출
Claude에 차트 및 그래프를 전달하는 가장 좋은 방법은 vision 기능과 PDF 지원 기능을 활용하는 것입니다. 즉, 차트 또는 그래프의 PDF 문서와 그에 대한 텍스트 질문을 Claude에 제공합니다.

현재는 `claude-sonnet-4-6`만 PDF 기능을 지원합니다. 이 기능은 아직 beta이므로 `pdfs-2024-09-25` beta header를 제공해야 합니다.

In [ ]:
# Anthropic client 설치 및 생성.
%pip install anthropic

In [2]:
import base64

from anthropic import Anthropic

# PDF 지원이 beta인 동안 올바른 beta header를 전달해야 합니다
client = Anthropic(default_headers={"anthropic-beta": "pdfs-2024-09-25"})
# 현재는 claude-sonnet-4-6만 PDF를 지원합니다
MODEL_NAME = "claude-sonnet-4-6"

In [37]:
# 유용한 보조 함수 만들기.
def get_completion(messages):
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=8192, temperature=0, messages=messages
    )
    return response.content[0].text

In [12]:
# 시작하려면 PDF가 필요합니다. cvna_2021_annual_report.pdf에 있는 .pdf 문서를 사용합니다.
# 먼저 PDF를 읽고 base64로 인코딩합니다.
with open("./documents/cvna_2021_annual_report.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

간단한 질문과 함께 이 문서를 모델에 전달하는 방법을 살펴보겠습니다.

In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": "What's in this document? Answer in a single sentence."},
        ],
    }
]

print(get_completion(messages))

This is a page from Carvana's 2021 Annual Report showing four key metrics: retail units sold, total revenue, total markets at year end, and car vending machines, all displaying significant growth from 2014 to 2021.


꽤 좋습니다! 이제 더 유용한 질문을 해 보겠습니다.

In [15]:
questions = [
    "What was CVNA revenue in 2020?",
    "How many additional markets has Carvana added since 2014?",
    "What was 2016 revenue per retail unit sold?",
]

for index, question in enumerate(questions):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": base64_string,
                    },
                },
                {"type": "text", "text": question},
            ],
        }
    ]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
According to the graph showing Total Revenue ($M), Carvana's revenue in 2020 was $5,587 million (or approximately $5.59 billion).

----------Question 2----------
According to the "TOTAL MARKETS AT YEAR END" graph, Carvana started with 4 markets in 2014 and grew to 311 markets by 2021. Therefore, Carvana added 307 markets since 2014 (311 - 4 = 307 additional markets).

----------Question 3----------
Let me calculate this for you:

2016 Revenue: $365 million
2016 Retail Units Sold: 18,761 units

$365 million ÷ 18,761 units = $19,455 per unit (rounded to nearest dollar)

So in 2016, Carvana's revenue per retail unit sold was approximately $19,455.


보시다시피 Claude는 차트와 그래프에 대한 상당히 자세한 질문에 답할 수 있습니다. 다만 최대한 활용하는 데 도움이 되는 몇 가지 팁과 기법이 있습니다.
- 때때로 Claude의 산술 능력이 방해가 됩니다. 위의 세 번째 질문을 sampling하면 산술 계산을 잘못하여 종종 틀린 최종 답을 출력한다는 점을 알 수 있습니다. 이러한 실수를 하지 않도록 Claude에 calculator tool을 제공하는 것을 고려하세요.
- 매우 복잡한 차트 및 그래프에서는 기존 Chain of Thought에서 보았던 것과 비슷한 개선을 이끌어내기 위해 Claude에 "먼저 문서에서 보이는 모든 데이터 포인트를 설명하세요"라고 요청할 수 있습니다.
- Claude는 그룹이 많은 grouped bar chart처럼 정보를 전달하기 위해 많은 색상에 의존하는 차트를 때때로 어려워합니다. 먼저 HEX code로 그래프의 색상을 식별하도록 Claude에 요청하면 정확도를 높일 수 있습니다.

## 슬라이드 자료
Claude가 차트와 그래프의 달인이라는 것을 알았으니, 차트와 그래프의 진정한 무대인 슬라이드 자료로 확장하는 것은 당연합니다!

슬라이드는 금융 서비스를 포함한 여러 분야에서 중요한 정보원입니다. PyPDF와 같은 패키지를 사용하여 슬라이드 자료에서 텍스트를 추출할 *수는* 있지만, 차트/그래프가 많은 특성 때문에 모델이 실제로 필요한 정보에 접근하기 어려워져 좋은 선택이 아닌 경우가 많습니다.

따라서 PDF 지원 기능이 훌륭한 대안이 될 수 있습니다. 이 기능은 PDF 문서를 처리할 때 추출한 텍스트와 vision을 모두 순서대로 사용합니다. 이 절에서는 Claude에서 PDF 문서를 사용하여 슬라이드 자료를 검토하는 방법과 이 접근 방식에서 흔히 발생하는 문제를 처리하는 방법을 살펴보겠습니다.

일반적인 슬라이드 자료를 claude에 전달하는 가장 좋은 방법은 PDF로 다운로드하여 Claude에 직접 제공하는 것입니다.

In [17]:
# 앞에서와 같은 방법으로 여러 페이지의 PDF 문서를 엽니다.
with open("./documents/twilio_q4_2023.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

In [18]:
# 이제 문서를 Claude에 직접 전달합니다. Claude는 문서의 텍스트와 시각적 요소를 모두 처리합니다.
question = "What was Twilio y/y revenue growth for fiscal year 2023?"
content = [
    {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": base64_string},
    },
    {"type": "text", "text": question},
]

messages = [{"role": "user", "content": content}]

print(get_completion(messages))

According to the financial results shown in the presentation, Twilio's year-over-year revenue growth for fiscal year 2023 was 9%. This can be found in the "Total Company Results Highlights" section, which shows FY 2023 revenue growth of 9%.


이 접근 방식은 시작하기에 좋은 방법이며, 일부 사용 사례에서는 최고의 성능을 제공합니다. 그러나 몇 가지 제한이 있습니다.
- 요청에 제공하는 모든 문서의 페이지를 합쳐 총 100페이지만 포함할 수 있습니다(시간이 지나면서 이 제한을 늘릴 계획입니다).
- 슬라이드 내용을 RAG의 일부로 사용하는 경우 multimodal PDF를 embedding에 도입하면 문제가 발생할 수 있습니다

다행히 Claude의 vision 기능을 활용하면 일반적인 pdf 텍스트 추출보다 훨씬 더 높은 품질로 슬라이드 자료를 **텍스트 형태**로 표현할 수 있습니다.

이를 위한 가장 좋은 방법은 현재 슬라이드와 이전 설명을 전달하면서 Claude에 처음부터 끝까지 순서대로 자료를 설명하도록 요청하는 것이라고 판단합니다. 방법을 살펴보겠습니다.

In [41]:
# 슬라이드 자료를 설명할 prompt를 정의합니다. 자료의 특성에 따라 prompt를 조정하되 구조는 대체로 유지합니다.
prompt = """
You are the Twilio CFO, narrating your Q4 2023 earnings presentation.

The entire earnings presentation document is provided to you.
Please narrate this presentation from Twilio's Q4 2023 Earnings as if you were the presenter. Do not talk about any things, especially acronyms, if you are not exactly sure you know what they mean.

Do not leave any details un-narrated as some of your viewers are vision-impaired, so if you don't narrate every number they won't know the number.

Structure your response like this:
<narration>
    <page_narration id=1>
    [Your narration for page 1]
    </page_narration>

    <page_narration id=2>
    [Your narration for page 2]
    </page_narration>

    ... and so on for each page
</narration>

Use excruciating detail for each page, ensuring you describe every visual element and number present. Show the full response in a single message.
"""
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# 이제 prompt로 전체 자료를 설명합니다. 실행에는 몇 분이 걸릴 수 있습니다(종종 최대 10분).
completion = get_completion(messages)

In [42]:
import re

# 다음으로 regex로 Claude의 응답을 parsing합니다
pattern = r"<narration>(.*?)</narration>"
match = re.search(pattern, completion.strip(), re.DOTALL)
if match:
    narration = match.group(1)
else:
    raise ValueError("No narration available. Likely due to the model response being truncated.")

이제 텍스트 기반 설명이 있으므로(완벽함과는 거리가 있지만 꽤 좋습니다), 이 자료를 어떤 텍스트 전용 workflow에서든 사용할 수 있습니다. Vector search도 포함됩니다!

마지막 sanity check로, 설명만을 사용하는 설정에 몇 가지 질문을 해 보겠습니다!

In [43]:
questions = [
    "What percentage of q4 total revenue was the Segment business line?",
    "Has the rate of growth of quarterly revenue been increasing or decreasing? Give just an answer.",
    "What was acquisition revenue for the year ended december 31, 2023 (including negative revenues)?",
]

for index, question in enumerate(questions):
    prompt = f"""You are an expert financial analyst analyzing a transcript of Twilio's earnings call.
Here is the transcript:
<transcript>
{narration}
</transcript>

Please answer the following question:
<question>
{question}
</question>"""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
Let me calculate this:

Segment revenue in Q4 2023: $75 million
Total revenue in Q4 2023: $1,076 million

$75M ÷ $1,076M = 0.0697 or approximately 7%

Therefore, the Segment business line represented approximately 7% of Twilio's total Q4 2023 revenue.

----------Question 2----------
Decreasing. The transcript shows Q4 2023 revenue growth was 5% year-over-year, while for the full year 2023 revenue growth was 9% year-over-year, indicating a slowing growth rate. Additionally, the Q1 2024 guidance projects even lower growth of 2-3% year-over-year, confirming the declining trend.

----------Question 3----------
Let me help calculate the acquisition revenue for 2023.

From the transcript, we can see:
- Total revenue for 2023: $4,154 million
- Organic revenue for 2023: $4,146 million

Therefore, acquisition revenue would be:
Total Revenue - Organic Revenue = $4,154M - $4,146M = $8 million

So the acquisition revenue for the year ended December 31, 2023 was $8 m

좋아 보입니다! 이 기법들을 활용하면 슬라이드 자료처럼 차트와 그래프가 많은 콘텐츠에 모델을 적용할 준비가 된 것입니다.